In [1]:
import sys
from numbers import Real
from pathlib import Path
from html import escape

import pandas as pd
from IPython.display import HTML, display

sys.path.append("../")

from evaluation.classes import AnswerRelevanceWrapper
from evaluation.eval import answer_relevance, load_calibration_tests
from evaluation.reporting import save_answer_relevance_calibration_report

/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tests = load_calibration_tests()

In [3]:
rows = []
threshold = 0.5

for test in tests:
    score = await answer_relevance(
        AnswerRelevanceWrapper(
            question=test.question,
            answer=test.candidate_answer,
        )
    )
    answer_relevance_score = float(score) if isinstance(score, Real) else 0.0
    predicted_is_relevant = int(answer_relevance_score >= threshold)
    expected_is_relevant = int(test.relevance_label)

    rows.append({
        "question": test.question,
        "answer_type": test.answer_type,
        "answer_relevance_score": answer_relevance_score,
        "predicted_is_relevant": predicted_is_relevant,
        "expected_is_relevant": expected_is_relevant,
        "label_match": int(predicted_is_relevant == expected_is_relevant),
    })

report_df = pd.DataFrame(rows)
report_path = save_answer_relevance_calibration_report(
    report_df=report_df,
    output_path=Path("../reports/faq_answer_relevance_calibration_report.md"),
    threshold=threshold,
)

display(HTML(f"<p><strong>Report saved to:</strong> {escape(str(report_path))}</p>"))
report_df[[
    "question",
    "answer_type",
    "answer_relevance_score",
    "predicted_is_relevant",
    "expected_is_relevant",
    "label_match",
]].round({"answer_relevance_score": 3})

,question,answer_type,answer_relevance_score,predicted_is_relevant,expected_is_relevant,label_match
0,Will I get a full refund including shipping if...,adversarial,0.833,1,1,1
1,Will I get a full refund including shipping if...,bad,0.896,1,0,0
2,Will I get a full refund including shipping if...,good,0.524,1,1,1
3,Can I use multiple delivery addresses for one ...,adversarial,0.832,1,1,1
4,Can I use multiple delivery addresses for one ...,bad,0.948,1,0,0
...,...,...,...,...,...,...
145,How can I change the delivery date before my o...,bad,0.928,1,0,0
146,How can I change the delivery date before my o...,good,0.469,0,1,0
147,What should I do if my order delivery is delay...,adversarial,1.000,1,1,1
148,What should I do if my order delivery is delay...,bad,0.974,1,0,0
